In [1]:
import requests  # Para realizar peticiones HTTP a la API externa
import pandas as pd  # Para la manipulación y análisis de datos en tablas (DataFrames)
import os  # Para interactuar con el sistema operativo (calcular tamaño de archivos, rutas, etc.)

# --- 1. CONFIGURACIÓN Y PETICIÓN A LA API ---

# Endpoint oficial de la NHTSA para obtener todas las marcas de vehículos registrados
url_vehiculos = "https://vpic.nhtsa.dot.gov/api/vehicles/getallmakes?format=json"

# Se define un User-Agent en las cabeceras para identificarnos de manera ética
headers = {
    'User-Agent': 'Mozilla/5.0 (educational-project/1.0)'
}

print("Consultando la API de la NHTSA...")
# Realiza la petición GET con un tiempo límite de espera de 15 segundos (timeout)
response = requests.get(url_vehiculos, headers=headers, timeout=15)

# --- 2. VALIDACIÓN DE LA RESPUESTA ---

# Si el servidor responde con un código 200 (OK), la petición fue exitosa
if response.status_code == 200:
    datos = response.json()  # Convierte la respuesta en un diccionario/JSON de Python
    print(f" Conexión exitosa. Código de estado: {response.status_code}")
    print(f"   Total de registros recibidos: {datos['Count']}")
else:
    # Si ocurre un error, detiene la ejecución del script mostrando el código de error
    raise Exception(f" Error al consultar la API. Código: {response.status_code}")

# --- 3. CREACIÓN DEL DATAFRAME ---

# Extrae la lista de marcas que viene en la clave 'Results' del JSON y la convierte en una tabla de Pandas
df_vehiculos = pd.DataFrame(datos['Results'])

# Imprime la cantidad inicial de (filas, columnas) obtenidas
print(f"Dimensiones: {df_vehiculos.shape[0]} filas × {df_vehiculos.shape[1]} columnas")

# --- 4. LIMPIEZA Y TRANSFORMACIÓN DE DATOS (ETL) ---

# 4.1 Renombrar columnas a español para mejorar la legibilidad del proyecto
df_vehiculos = df_vehiculos.rename(columns={
    'Make_ID': 'ID_Marca',
    'Make_Name': 'Nombre_Marca'
})

# 4.2 Eliminar duplicados por ID (asegura que cada ID de marca aparezca solo una vez)
filas_antes = len(df_vehiculos)
df_vehiculos = df_vehiculos.drop_duplicates(subset='ID_Marca')
print(f"Duplicados eliminados: {filas_antes - len(df_vehiculos)}")

# 4.3 Eliminar filas con valores nulos (NaN) en cualquier columna
df_vehiculos = df_vehiculos.dropna()

# 4.4 Limpiar espacios en blanco innecesarios al inicio o al final del texto (ej. " FORD " -> "FORD")
df_vehiculos['Nombre_Marca'] = df_vehiculos['Nombre_Marca'].str.strip()

# 4.5 Formato título: coloca la primera letra de cada palabra en mayúscula (ej. "TOYOTA MOTOR" -> "Toyota Motor")
df_vehiculos['Nombre_Marca'] = df_vehiculos['Nombre_Marca'].str.title()

# 4.6 Eliminar filas donde el nombre haya quedado vacío tras la limpieza
df_vehiculos = df_vehiculos[df_vehiculos['Nombre_Marca'].str.len() > 0]

print(f" Limpieza completada. Registros válidos: {len(df_vehiculos)}")

# --- 5. ENRIQUECIMIENTO DE LOS DATOS (INGENIERÍA DE CARACTERÍSTICAS) ---

# Crea una columna con la cantidad total de caracteres de cada marca
df_vehiculos['Largo_Nombre'] = df_vehiculos['Nombre_Marca'].str.len()

# Determina el número de palabras separando el texto por espacios en blanco
df_vehiculos['Num_Palabras'] = df_vehiculos['Nombre_Marca'].str.split().str.len()

# Función auxiliar para segmentar las marcas según la longitud de su nombre
def clasificar_nombre(largo):
    if largo <= 4:
        return 'Corto'
    elif largo <= 9:
        return 'Medio'
    else:
        return 'Largo'

# Aplica la función fila por fila para crear una columna categórica
df_vehiculos['Categoria_Nombre'] = df_vehiculos['Largo_Nombre'].apply(clasificar_nombre)

# Reordena las columnas en una estructura lógica y resetea el índice de la tabla para que sea consecutivo (0, 1, 2...)
df_vehiculos = df_vehiculos[['ID_Marca', 'Nombre_Marca', 'Largo_Nombre', 'Num_Palabras', 'Categoria_Nombre']]
df_vehiculos = df_vehiculos.reset_index(drop=True)

# Muestra en pantalla las primeras 10 filas del resultado final
print("Vista previa del dataset final transformado:")
display(df_vehiculos.head(10))

# --- 6. ANÁLISIS ESTADÍSTICO Y RESUMEN ---

print("=" * 45)
print("        RESUMEN DEL DATASET FINAL")
print("=" * 45)
print(f"  Total de marcas registradas : {len(df_vehiculos)}")
print(f"  Promedio de caracteres      : {df_vehiculos['Largo_Nombre'].mean():.1f}")

# .idxmin() e .idxmax() encuentran la posición de los nombres con menor y mayor número de caracteres
print(f"  Nombre más corto            : {df_vehiculos.loc[df_vehiculos['Largo_Nombre'].idxmin(), 'Nombre_Marca']}")
print(f"  Nombre más largo            : {df_vehiculos.loc[df_vehiculos['Largo_Nombre'].idxmax(), 'Nombre_Marca']}")

print(f"\nDistribución por Categoría de Nombre:")
# Cuenta cuántas marcas entran en 'Corto', 'Medio' y 'Largo'
print(df_vehiculos['Categoria_Nombre'].value_counts().to_string())

print(f"\nTipos de datos finales:")
# Muestra un pequeño resumen en forma de tabla con el conteo de categorías
display(df_vehiculos['Categoria_Nombre'].value_counts().reset_index().rename(columns={'count': 'Total'}))

# --- 7. EXPORTACIÓN A ARCHIVO CSV ---

nombre_archivo = "marcas_vehiculos_nhtsa.csv"

# Guarda la información procesada en el disco local
df_vehiculos.to_csv(
    nombre_archivo,
    index=False,           # No guardar la columna de índices internos de Pandas
    encoding='utf-8-sig',  # UTF-8 con BOM: formato ideal para que Excel lea tildes y caracteres especiales sin romperse
    sep=',',               # Separador por comas estándar
    decimal='.'            # Indica que el punto es el separador de decimales internacional
)

# --- 8. VERIFICACIÓN FINAL DEL ARCHIVO ---

# Obtiene el peso del archivo en bytes, lo divide entre 1024 para pasarlo a Kilobytes (KB)
tamanio_kb = os.path.getsize(nombre_archivo) / 1024

print("=" * 45)
print(f" Archivo CSV exportado exitosamente")
print("=" * 45)
print(f"  Nombre          : {nombre_archivo}")
print(f"  Filas exportadas: {len(df_vehiculos)}")
print(f"  Columnas        : {list(df_vehiculos.columns)}")
print(f"  Tamaño          : {tamanio_kb:.1f} KB")
print(f"  Ubicación       : {os.path.abspath(nombre_archivo)}") # Muestra la ruta completa del archivo en tu PC

Consultando la API de la NHTSA...
 Conexión exitosa. Código de estado: 200
   Total de registros recibidos: 12260
Dimensiones: 12260 filas × 2 columnas
Duplicados eliminados: 0
 Limpieza completada. Registros válidos: 12260
Vista previa del dataset final transformado:


,ID_Marca,Nombre_Marca,Largo_Nombre,Num_Palabras,Categoria_Nombre
0,12858,#1 Alpine Customs,17,3,Largo
1,4877,"1/Off Kustoms, Llc",18,3,Largo
2,11257,"102 Ironworks, Inc.",19,3,Largo
3,12255,12832429 Canada Inc.,20,3,Largo
4,13053,137 Industries Inc.,19,3,Largo
5,6387,17 Creek Enterprises,20,3,Largo
6,12948,1955 Custom Belair,18,3,Largo
7,9172,"1M Custom Car Transports, Inc.",30,5,Largo
8,6124,1St Choice Manufacturing Inc,28,4,Largo
9,12972,2 Golden Eagles,15,3,Largo


        RESUMEN DEL DATASET FINAL
  Total de marcas registradas : 12260
  Promedio de caracteres      : 17.3
  Nombre más corto            : Z
  Nombre más largo            : Irbit Motorworks Of America, Inc Hdqtrs For Irbitski Mototsicletny Zavod

Distribución por Categoría de Nombre:
Categoria_Nombre
Largo    9549
Medio    2138
Corto     573

Tipos de datos finales:


,Categoria_Nombre,Total
0,Largo,9549
1,Medio,2138
2,Corto,573


 Archivo CSV exportado exitosamente
  Nombre          : marcas_vehiculos_nhtsa.csv
  Filas exportadas: 12260
  Columnas        : ['ID_Marca', 'Nombre_Marca', 'Largo_Nombre', 'Num_Palabras', 'Categoria_Nombre']
  Tamaño          : 425.5 KB
  Ubicación       : c:\Users\Personal\Documents\BIT\Taller 03\marcas_vehiculos_nhtsa.csv
